# Вино
В этом блокноте будем определять качество вина.
В частности, будем исследовать модель KNN.

[ссылка на датасет](https://www.kaggle.com/datasets/yasserh/wine-quality-dataset)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/wine-quality-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sb.set_theme(style="darkgrid", context="notebook", palette="deep")

df = pd.read_csv(path + "/WineQT.csv")

### Быстрый обзор

Первые пять записей

In [ ]:
df.head()

Разрешение фрейма

In [ ]:
df.shape

Распределение классов

In [ ]:
df['quality'].value_counts()

Наблюдается сильный дисбаланс классов и это большая проблема.

In [ ]:
plt.figure(figsize=(8,5))
sb.countplot(x='quality', data=df)
plt.title('Распределение классов')
plt.show()

Краткая информация о фрейме

In [ ]:
df.describe()

как видим, дисбаланс подтвердился, чтобы классификация хорошо работала, нужно будет стратифицировать тестовую выборку, либо разбить классы на две группы: хорошие вина и плохие. Получится бинарная классификация. Однако цель данного блокнота - исследовать knn.

In [ ]:
df.info()

### Ищем дубликаты

In [ ]:
df.duplicated().sum()

### Асимметрия и эксцес

In [ ]:
df.skew()

Видим, что справа у нас длинный хвост, особенно по хлоридам. Возможно, от них зависит качество вина.

In [ ]:
df.kurt()

У колонок chlorides, residual sugar, sulphates наблюдаются экстремальные выбросы, нужно обработать датасет, тк KNN [ алгоритм ] чувствителен к выбросам.

Построим матрицу корреляций.

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
sb.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Тепловая карта корреляций')
plt.show()

Как видим, больше всего на качество влияют:
alcohol
volatile_acidity
sulphates
citric_acid

У нас есть признаки, которые коррелируют между собой и их нужно убрать.
fixed_acidity, density, free_sulfur_dioxide


### Подготовка данных

Уберем ложные корреляции.

In [ ]:
df_prep = df.drop(['fixed acidity', 'density', 'free sulfur dioxide', 'Id'], axis=1)

### Разбиение на тесты

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
train_data, test_data = train_test_split(df_prep, test_size=0.2, stratify=df['quality'], random_state=42)

In [ ]:
X_train, y_train = train_data.drop('quality', axis=1),train_data['quality']
X_test, y_test = test_data.drop('quality', axis=1),test_data['quality']

### Масштабирование

**Почему это необходимо?**

Различные признаки имеют различные единицы измерений. Если мы не отмасштабируем данные, то изменение диоксида серы на 1 единицу будет влиять на расстояние намного сильнее, чем изменение хлоридов, и поэтому модель решит, что хлориды не важны.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


**Почему нельзя подбирать параметры на тестовой выборке?**

Потому что мы просто подгоним параметры под результаты тестов, произойдет утечка данных, и в реальных случаях модель будет ошибаться. По той же причине нельзя фиттить тестовые признаки, тк mean даёт информацию о данных, что является утечкой.

### Обучение модели

Тут масштабированные данные.

Исследуем влияние количества соседей и типов метрик. (тут подбор вручную)

In [ ]:
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_neighbors=4, metric='euclidean', weights='distance')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

По евклидовой метрике самый лучший для угадывания редких классов вариант оказался с 3 соседями и весами по дистанции, тк при таких параметрах больше всего угадано класса 4.

In [ ]:
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_neighbors=2, metric='manhattan', weights='distance')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

Слишком мало редких классов, чтобы они нормально определялись.

Выводы:
* Количество соседей - если мало, то больше вероятность найти редкий класс, но и больше вероятность переобучения.

* Для редких классов лучше использовать веса по дистанции, тк если будет несколько классов, и самый редкий будет ближе всего, то он будет иметь больше влияния.

* Для редких классов лучше показала себя евклидова метрика.

In [ ]:
param_grid = {
    'n_neighbors': range(1, 30),           
    'weights': ['uniform', 'distance'],    
    'metric': ['euclidean', 'manhattan', 'minkowski', 'cosine']
}

knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_macro')
grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший результат: {grid_search.best_score_}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

Дисбаланс сильно сказывается на точности модели. Как видим, KNN не определяет редкие классы, только три самых распространенных, даже при использовании f1_macro скоринге, который ставит в приоритет редкие классы.

Поэтому сконцентрируемся на определении массовых классов с наилучшей точностью.

In [ ]:
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_micro')
grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший результат: {grid_search.best_score_}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

66% точности набивается только за счет угадывания классов 5, 6, 7, тк их больше всего в выборке.

***Если данные не масштабировать***

In [ ]:
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_micro')
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший результат: {grid_search.best_score_}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

Точность сильно ниже по сравнению с нормализованными данными.

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
cmd.plot(cmap='Blues')
plt.title('Матрица ошибок для качества вина')
plt.show()

Видим, что модель угадывает классы 5 и 6, и немного 7, но ошибается. И при этом модель не использует метки 3, 4 и 8. Вначале мы и убедились, что этих классов меньше всего.